# Conn2Conn — W&B Results Scraper

Pulls best-trial runs from W&B, builds status/metric tables, and integrates with local `ray_results` artifacts.

**Sections**
1. Setup
2. Define experiment spec
3. Fetch W&B records
4. Debug: status table (which seeds completed / failed)
5. Metrics table (test demeaned Pearson, mean ± std across seeds)
6. Raw per-seed DataFrame
7. Local artifact enrichment (identifiability, PCA structure)
8. Plots

In [7]:
import sys
from pathlib import Path

REPO_ROOT = Path('/scratch/asr655/neuroinformatics/Conn2Conn')
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import wandb

from results.results_scraper import (
    build_experiment_records,
    records_to_df,
    build_status_table,
    build_metric_table,
    enrich_records_with_local,
    load_local_artifact_df,
)

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

# Log into W&B (reads from ~/.netrc automatically).
wandb.login()

True

## 1. Experiment Spec
Edit this cell to change which models / sources / seeds to pull.

In [8]:
# ── SC vs SC_r2t vs SC+SC_r2t experiment ──────────────────────────────────

# Models that support all three source conditions.
MULTIMODAL_MODELS = [
    'CrossModal_PCA_PLS',
    #'CrossModal_PCA_PLS_learnable',
]

# Single-source models (SC and SC_r2t only — no SC+SC_r2t).
SINGLE_SOURCE_MODELS = [
    'CrossModalPCA',
    'CrossModal_PLS_SVD',
]

SOURCES_MULTI  = ['SC', 'SC_r2t', 'SC+SC_r2t']
SOURCES_SINGLE = ['SC', 'SC_r2t']
SEEDS = [0, 1, 2, 3, 4]

# Primary metric for ranking / table display.
PRIMARY_METRIC = 'demeaned_pearson'   # key in RunRecord.test_metrics
SECONDARY_METRIC = 'mse'

## 2. Fetch W&B Records

`build_experiment_records` issues one W&B API call per model to fetch all
`best_trial_report` runs, then cross-joins against the expected
(model × source × seed) matrix.  Set `count_trials=False` to skip the
per-sweep trial-count calls if you only care about metrics.

In [11]:
import wandb
from results.results_scraper import (
    build_experiment_records,
    records_to_df,
    build_status_table,
    build_metric_table,
)


In [ ]:
records_multi = build_experiment_records(
    models=MULTIMODAL_MODELS,
    sources=SOURCES_MULTI,
    seeds=SEEDS,
    count_trials=True,
    verbose=True,
)

records_single = build_experiment_records(
    models=SINGLE_SOURCE_MODELS,
    sources=SOURCES_SINGLE,
    seeds=SEEDS,
    count_trials=True,
    verbose=True,
)

all_records = records_multi + records_single
n_complete = sum(r.status == 'complete' for r in all_records)
n_missing  = sum(r.status == 'missing'  for r in all_records)
print(f"\nTotal records: {len(all_records)}  (complete={n_complete}, missing={n_missing})")

Fetching W&B runs for CrossModal_PCA_PLS…
  ✗ MISSING  CrossModal_PCA_PLS | src=SC | seed=0
  ✗ MISSING  CrossModal_PCA_PLS | src=SC | seed=1
  ✗ MISSING  CrossModal_PCA_PLS | src=SC | seed=2
  ✗ MISSING  CrossModal_PCA_PLS | src=SC | seed=3
  ✗ MISSING  CrossModal_PCA_PLS | src=SC | seed=4
  ✗ MISSING  CrossModal_PCA_PLS | src=SC_r2t | seed=0
  ✗ MISSING  CrossModal_PCA_PLS | src=SC_r2t | seed=1
  ✗ MISSING  CrossModal_PCA_PLS | src=SC_r2t | seed=2
  ✗ MISSING  CrossModal_PCA_PLS | src=SC_r2t | seed=3
  ✗ MISSING  CrossModal_PCA_PLS | src=SC_r2t | seed=4
  ✗ MISSING  CrossModal_PCA_PLS | src=SC+SC_r2t | seed=0
  ✗ MISSING  CrossModal_PCA_PLS | src=SC+SC_r2t | seed=1
  ✗ MISSING  CrossModal_PCA_PLS | src=SC+SC_r2t | seed=2
  ✗ MISSING  CrossModal_PCA_PLS | src=SC+SC_r2t | seed=3
  ✗ MISSING  CrossModal_PCA_PLS | src=SC+SC_r2t | seed=4
Fetching W&B runs for CrossModalPCA…
Fetching W&B runs for CrossModal_PLS_SVD…


In [ ]:
## Diagnostic: inspect raw W&B config for one complete run
# Helps verify that source / seed are being parsed correctly from the
# nested config structure that best-trial runs log.

import wandb as _wandb
from results.results_scraper import fetch_best_trial_runs, _source_from_config, _seed_from_config

_diag_model = (MULTIMODAL_MODELS + SINGLE_SOURCE_MODELS)[0]
_raw_runs = fetch_best_trial_runs(_diag_model)
print(f"Found {len(_raw_runs)} best-trial run(s) for '{_diag_model}'")

if _raw_runs:
    r0 = _raw_runs[0]
    _cfg = dict(r0.config)
    print(f"\nRun: {r0.name}  |  group: {r0.group}  |  tags: {r0.tags}")
    print(f"\nTop-level config keys: {list(_cfg.keys())}")
    if 'data' in _cfg:
        print(f"  config['data'] = {_cfg['data']}")
    else:
        # flat key check
        for k in ('data.source', 'source', 'data.shuffle_seed', 'shuffle_seed'):
            if k in _cfg:
                print(f"  config['{k}'] = {_cfg[k]}")
    print(f"\nParsed → source='{_source_from_config(_cfg)}'  seed={_seed_from_config(_cfg)}")
    print(f"\nW&B summary keys (subset):")
    _s = dict(r0.summary)
    for k in sorted(_s):
        if any(k.startswith(p) for p in ('val_', 'train_', 'eval_test/')):
            print(f"  {k}: {_s[k]}")

## 3. Debug — Status Table

Shows which (model × source × seed) cells completed and how many tune
trials were explored. **Cells marked `✗ MISSING` indicate failed or
never-run experiments.**

In [ ]:
status_multi  = build_status_table(records_multi)
status_single = build_status_table(records_single)

print('── Multimodal models ──')
display(status_multi)

print('\n── Single-source models ──')
display(status_single)

## 4. Metrics Table — Test Demeaned Pearson

Aggregated across seeds (mean ± std).  Only seeds with `status='complete'`
contribute.  Missing cells show `NaN`.

In [ ]:
metric_multi  = build_metric_table(records_multi,  metric=PRIMARY_METRIC, agg='mean±std')
metric_single = build_metric_table(records_single, metric=PRIMARY_METRIC, agg='mean±std')

# Combine into one table for display.
metric_table = pd.concat([metric_multi, metric_single]).sort_index()

print(f'Test {PRIMARY_METRIC} — mean ± std across seeds')
display(metric_table)

In [ ]:
# Secondary metric for cross-check.
mse_table = pd.concat([
    build_metric_table(records_multi,  metric=SECONDARY_METRIC, agg='mean±std'),
    build_metric_table(records_single, metric=SECONDARY_METRIC, agg='mean±std'),
]).sort_index()

print(f'Test {SECONDARY_METRIC} — mean ± std across seeds')
display(mse_table)

## 5. Raw Per-Seed DataFrame

Flat table with one row per (model, source, seed).  Useful for per-seed
inspection and feeding into downstream plotting.

In [ ]:
df = records_to_df(all_records)
print(f'Shape: {df.shape}')
display(df[['model', 'source', 'seed', 'status', 'n_tune_trials',
            'test_demeaned_pearson', 'test_mse', 'val_demeaned_r']].sort_values(
    ['model', 'source', 'seed']))

In [ ]:
# Inspect failed / missing seeds in detail.
missing_df = df[df['status'] == 'missing']
if missing_df.empty:
    print('All expected runs completed.')
else:
    print(f'{len(missing_df)} missing runs:')
    display(missing_df[['model', 'source', 'seed']])

## 6. Local Artifact Enrichment

Merges full local `metrics_final.json` (identifiability heatmaps, violin,
PCA structure, Hungarian matching) into each RunRecord's `test_metrics`.
Only runs with a `local_artifact_path` are enriched; others are unchanged.

In [ ]:
all_records_enriched = enrich_records_with_local(all_records)

n_enriched = sum(
    1 for r in all_records_enriched
    if r.status == 'complete' and r.local_artifact_path
)
print(f'Enriched {n_enriched} / {sum(r.status=="complete" for r in all_records_enriched)} complete runs with local metrics.')

In [ ]:
# Full local artifact DataFrame (mirrors local_results_utils.load_local_results).
local_df = load_local_artifact_df(all_records_enriched)
print(f'Local artifact DataFrame: {local_df.shape}')
if not local_df.empty:
    display(local_df[['model', 'source', 'seed',
                       'test_base_metrics_demeaned_pearson',
                       'test_heatmaps_demeaned_top1_acc',
                       'test_heatmaps_demeaned_avg_rank_percentile']].sort_values(
        ['model', 'source', 'seed']))

## 7. Plots

In [ ]:
def plot_metric_by_source(
    records,
    metric='demeaned_pearson',
    title=None,
    figsize=(14, 5),
):
    """
    Grouped bar chart: one group per model, bars per source.
    Each bar shows mean across seeds; error bar shows ±1 std.
    Missing cells are skipped.
    """
    complete = [r for r in records if r.status == 'complete']
    if not complete:
        print('No complete records to plot.')
        return

    models  = sorted({r.model_name for r in complete})
    sources = sorted({r.source     for r in complete})

    x = np.arange(len(models))
    width = 0.8 / len(sources)
    offsets = np.linspace(-0.4 + width/2, 0.4 - width/2, len(sources))

    fig, ax = plt.subplots(figsize=figsize)
    for i, src in enumerate(sources):
        means, stds = [], []
        for mdl in models:
            vals = [
                r.test_metrics.get(metric)
                for r in complete
                if r.model_name == mdl and r.source == src
                and r.test_metrics.get(metric) is not None
            ]
            means.append(np.mean(vals) if vals else np.nan)
            stds.append(np.std(vals)  if vals else 0.0)
        ax.bar(x + offsets[i], means, width, yerr=stds,
               label=src, capsize=4, alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels(models, rotation=25, ha='right')
    ax.set_ylabel(metric)
    ax.set_title(title or f'Test {metric} by source (mean ± std across seeds)')
    ax.legend(title='Source')
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.4f'))
    plt.tight_layout()
    plt.show()


plot_metric_by_source(all_records, metric=PRIMARY_METRIC)
plot_metric_by_source(all_records, metric=SECONDARY_METRIC)

In [ ]:
def plot_seed_variance(
    records,
    metric='demeaned_pearson',
    source_filter=None,
    figsize=(14, 5),
):
    """
    Box plot: one box per (model, source) showing per-seed variance.
    """
    complete = [
        r for r in records
        if r.status == 'complete'
        and (source_filter is None or r.source in source_filter)
        and r.test_metrics.get(metric) is not None
    ]
    if not complete:
        print('No complete records to plot.')
        return

    from collections import defaultdict
    groups = defaultdict(list)
    for r in complete:
        key = f"{r.model_name}\n{r.source}"
        groups[key].append(r.test_metrics[metric])

    labels = sorted(groups.keys())
    data   = [groups[k] for k in labels]

    fig, ax = plt.subplots(figsize=figsize)
    ax.boxplot(data, labels=labels, patch_artist=True)
    ax.set_ylabel(metric)
    ax.set_title(f'Per-seed variance in test {metric}')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()


plot_seed_variance(all_records, metric=PRIMARY_METRIC)
plot_seed_variance(all_records, metric=PRIMARY_METRIC, source_filter=['SC', 'SC_r2t', 'SC+SC_r2t'])

In [ ]:
# ── Identifiability metrics (requires local artifact enrichment) ──────────
if not local_df.empty and 'test_heatmaps_demeaned_top1_acc' in local_df.columns:
    plot_df = local_df.copy()
    plot_df['label'] = plot_df['model'] + ' | ' + plot_df['source']

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, col, ylabel in zip(
        axes,
        ['test_heatmaps_demeaned_top1_acc', 'test_heatmaps_demeaned_avg_rank_percentile'],
        ['Demeaned Top-1 Acc', 'Demeaned Avg Rank %ile'],
    ):
        if col not in plot_df.columns:
            continue
        grp = plot_df.groupby(['model', 'source'])[col]
        means = grp.mean().reset_index()
        means['label'] = means['model'] + '\n' + means['source']
        means = means.sort_values(col, ascending=False)
        ax.bar(means['label'], means[col], alpha=0.85)
        ax.set_ylabel(ylabel)
        ax.set_title(ylabel)
        ax.tick_params(axis='x', rotation=30)
    plt.tight_layout()
    plt.show()
else:
    print('Local artifact enrichment not available — skipping identifiability plots.')